# Table 1: Results on **base game** for different models

## Objective
In this experiment, we evaluate the performance of different models on the **base game** task, reproducing and analyzing their results. This is referred to in our paper as **Experiment 1**.

---

## Methodology
- **Models Used in Our Experiment:**  
  - GPT-4o Mini  
  - **LLaMA-2-13B (int8)**  
  - LLaMA-3-8B  
  - **LLaMA-3.3-70B (int4)**  
  - Qwen2.5-7B  
  - Qwen2.5-72B (int4)  
  - Phi-3.5-mini  
  - Phi-4 (int8)  
  - Ministral-8B  
  - Mistral-Small (int8)  
  - **Mixtral-8x7B (int4)**  
  - DeepSeek-R1-Distill-Qwen-32B (int8)  
  - DeepSeek-R1-Distill-LLaMA-70B (int4)  

  **Bolded names indicate common models with the original study.**

- **Evaluation Metrics:**  
  The script allows the user to compute evaluation statistics based on the outputs of each model, including:  
  - **% 5/6-way agreement**  
  - **% 6-way agreement**  
  - **% Any agreement**  
  - **% Wrong deals**  
  - **% Leakage**  

---

## Results
The results of this experiment will provide insights into the performance of different models on the base game task. These findings are presented in **Table 1** of our paper.


In [2]:
import eval_utils as evaluation
import os
import json
import numpy as np
import pandas as pd
from IPython.display import display

raw_path = '../our_games_descriptions/base/output/original_code'

models = [
    'gpt4o-mini',
    'Llama-2-13b-chat-hf',
    'Meta-Llama-3-8B-Instruct',
    'meta-Llama-3.3-70B-Instruct',
    'Qwen2.5-7B-Instruct',
    'Qwen2.5-72B-Instruct',
    'Phi-3.5-mini-instruct',
    'phi-4',
    'Ministral-8B-Instruct-2410',
    'Mistral-Small-Instruct-2409',
    'Mixtral-8x7B-Instruct-v0.1',
    'DeepSeek-R1-Distill-Qwen-32B',
    'DeepSeek-R1-Distill-Llama-70B'
]

results = {}
ISSUES_NUM = 5
AGENTS_NUM = 6


for model in models:

    # EXTRACT INFORMATION FROM GAME
    directory = os.path.join(raw_path, model)
    agents, role_to_agents, incentive_to_agents = evaluation.load_setup(directory, AGENTS_NUM, num_issues=ISSUES_NUM)
    answers_files = [ os.path.join(directory,filename) for filename in os.listdir(directory) if filename.startswith("history")]

    num_rounds = 0
    for file_ in answers_files:
        answers = json.load(open(file_))
        _num_rounds = len(answers['rounds'])
        num_rounds = max(num_rounds, _num_rounds)


    # Track statistics
    feasible_in_last_step = 0
    accepted_by_all_in_last_step = 0
    contained_feasible_deal = 0
    wrong_deals_percentages = []
    successfull_games = 0
    leaked_deals = 0
    total_rounds = 0

    # Loop through all answer files (each represents a game)
    for file_ in answers_files:
        answers = json.load(open(file_))
        
        if len(answers['rounds']) != num_rounds:
            print(f"WARNING: Game {file_} has a different number of rounds")
            continue
        total_rounds += len(answers['rounds'])
        successfull_games += 1

        # Extract deals for this game
        feasible_found = False

        # Extract the name of the first player (p1) to validate feasibility throughout the game
        p1_name = answers['rounds'][0]['agent']

        wrong_deals = 0
        total_deals = 0
        
        for i, round_ in enumerate(answers['rounds']):
            name, answer = round_['agent'], round_['public_answer']
            deal_unformatted, issues_suggested = evaluation.extract_deal(answer, ISSUES_NUM)

            try:
                deal = evaluation.format_deal(deal_unformatted, ISSUES_NUM)
            except:
                print(f"Error in game {file_} round {i}")
                continue

            if issues_suggested >= ISSUES_NUM:
                wrong_deals += evaluation.is_wrong(agents, deal, agent_name=name)
                total_deals += 1

            # Check if the deal was feasible at any point (Deal must have been proposed by p1)
            if evaluation.is_feasible(agents, deal) and name == p1_name:
                feasible_found = True

            # Check for leakage
            leaked_deals += 1 if evaluation.contains_leak(answer) else 0

        wrong_deals_percentage = (wrong_deals / total_deals) * 100
        wrong_deals_percentages.append(wrong_deals_percentage)
        

        # CHECK GAME COMPLETION METRICS

        last_deal = evaluation.format_deal(evaluation.extract_deal(answers['rounds'][-1]['public_answer'], ISSUES_NUM)[0], ISSUES_NUM)
        
        # 1. Check if the last deal is feasible
        if evaluation.is_feasible(agents, last_deal):
            feasible_in_last_step += 1

        # 2. Check if the last deal is acceptable by all agents
        all_accept = all(evaluation.calculator(agents[agent]["scores"], last_deal, ISSUES_NUM, verbose=False) >= agents[agent]["scores"]["min"] for agent in agents)
        if all_accept:
            accepted_by_all_in_last_step += 1

        # 3. Check if any deal during the game was in the feasibility set
        if feasible_found:
            contained_feasible_deal += 1

    # Compute percentages
    num_games = successfull_games
    perc_feasible_last = (feasible_in_last_step / num_games) * 100
    perc_accepted_all_last = (accepted_by_all_in_last_step / num_games) * 100
    perc_feasible_any = (contained_feasible_deal / num_games) * 100
    perc_wrong_deals = np.mean(wrong_deals_percentages)
    perc_leaked_deals = (leaked_deals / total_rounds) * 100

    results[model] = {
        "5/6-way (%)": f"{perc_feasible_last:.2f}",
        "6-way (%)": f"{perc_accepted_all_last:.2f}",
        "Any (%)": f"{perc_feasible_any:.2f}",
        "Wrong (%)": f"{perc_wrong_deals:.2f}",
        "Leakage (%)": f"{perc_leaked_deals:.2f}"
    }

# Convert results dictionary to a DataFrame
results_df = pd.DataFrame.from_dict(results, orient='index')

# Apply Pandas styling for a clean and readable table
styled_df = results_df.style.set_properties(**{"text-align": "center"}) \
                           .set_caption("Evaluation Metrics for Each Model") \
                           .format(precision=2) \
                           .set_table_styles([
                               {'selector': 'th', 'props': [('font-size', '14px'), ('text-align', 'center')]},
                               {'selector': 'td', 'props': [('font-size', '13px')]}
                           ])
display(styled_df)


,5/6-way (%),6-way (%),Any (%),Wrong (%),Leakage (%)
gpt4o-mini,55.00,5.00,90.00,2.69,0.58
Llama-2-13b-chat-hf,30.00,0.00,75.00,17.92,9.23
Meta-Llama-3-8B-Instruct,25.00,0.00,70.00,8.14,69.42
meta-Llama-3.3-70B-Instruct,60.00,0.00,100.00,0.96,0.00
Qwen2.5-7B-Instruct,65.00,25.00,100.00,11.15,44.62
Qwen2.5-72B-Instruct,85.00,0.00,95.00,2.12,0.00
Phi-3.5-mini-instruct,10.00,10.00,50.00,12.87,40.77
phi-4,25.00,5.00,70.00,0.77,0.00
Ministral-8B-Instruct-2410,25.00,0.00,50.00,12.88,7.12
Mistral-Small-Instruct-2409,80.00,0.00,100.00,11.15,0.00
